# View Converted Cirq Circuits

Notebook for exploring circuits in `const_0_noiseless_lrb_cirq/`.

What this notebook does:
- lists available converted circuit files
- provides helper functions to load and inspect a selected circuit
- prints all operations and renders Cirq's SVG viewer for one file
- supports quick random spot-checks

## 1) Setup
Run this cell first to discover all `.cirq.pkl` files.

In [1]:
from pathlib import Path
import pickle
import random
from cirq.contrib.svg import SVGCircuit
from IPython.display import display

import cirq.circuits.circuit as _cirq_circuit

# Compatibility shim for legacy pickles saved with older Cirq internals.
if not hasattr(_cirq_circuit, "_PlacementCache"):
    class _PlacementCache(dict):
        pass

    _cirq_circuit._PlacementCache = _PlacementCache

# Compatibility shim for legacy LineQid pickle construction behavior.
try:
    from cirq.devices.line_qubit import LineQid

    def _lineqid_new(cls, x: int, dimension: int):
        obj = object.__new__(cls)
        obj._x = int(x)
        obj._dimension = int(dimension)
        return obj

    LineQid.__new__ = staticmethod(_lineqid_new)
except Exception as exc:
    print(f"[WARN] LineQid compatibility shim not applied: {exc}")

CIRQ_ROOT = Path("const_0_noiseless_lrb_cirq")
cirq_files = sorted(CIRQ_ROOT.rglob("*.cirq.pkl"))

print(f"CIRQ_ROOT: {CIRQ_ROOT.resolve()}")
print(f"Found {len(cirq_files)} circuit files")
print("First few files:")
for path in cirq_files[:10]:
    print(" -", path.relative_to(CIRQ_ROOT))


CIRQ_ROOT: C:\Users\Sohan XPS17\OneDrive - University of California, Davis\Documents\UC Davis\Kim Group\LBNL\LRB-Simulations-sdim\external\iqm_qpu_circuits\[[5,1,2]]_3 Qutrit Surface Code\X Stabilizer Terminal Check Circuits\const_0_noiseless_lrb_cirq
Found 270 circuit files
First few files:
 - 0\0.cirq.pkl
 - 0\1.cirq.pkl
 - 0\2.cirq.pkl
 - 0\3.cirq.pkl
 - 0\4.cirq.pkl
 - 0\5.cirq.pkl
 - 0\6.cirq.pkl
 - 0\7.cirq.pkl
 - 0\8.cirq.pkl
 - 1\0.cirq.pkl


## 2) Helper Functions
These helpers keep output readable and reusable.

In [2]:
def load_circuit(rel_path: str):
    path = CIRQ_ROOT / rel_path
    try:
        with path.open("rb") as f:
            circuit = pickle.load(f)
        return path, circuit, None
    except Exception as exc:
        return path, None, exc


def summarize_circuit(rel_path: str):
    path, circuit, err = load_circuit(rel_path)
    if err is not None or circuit is None:
        print(f"File: {path}")
        print(f"[ERROR] Failed to load pickle: {err}")
        return

    ops = list(circuit.all_operations())
    qudits = sorted(circuit.all_qubits(), key=lambda q: getattr(q, "x", 0))

    print(f"File: {path}")
    print(f"Qudits: {len(qudits)}")
    print(f"Operations: {len(ops)}")
    print(f"Depth (moments): {len(circuit)}")


def show_all_ops_and_view(rel_path: str):
    path, circuit, err = load_circuit(rel_path)
    if err is not None or circuit is None:
        print(f"File: {path}")
        print(f"[ERROR] Failed to load pickle: {err}")
        return

    ops = list(circuit.all_operations())

    summarize_circuit(rel_path)
    print("All operations:")
    for i, op in enumerate(ops, start=1):
        print(f"{i:4d}: {op}")

    print("\nCirq text diagram:")
    print(circuit)

    print("\nCirq SVG viewer:")
    display(SVGCircuit(circuit))


def preview_random(n: int = 3):
    chosen = random.sample(cirq_files, k=min(n, len(cirq_files)))
    for idx, path in enumerate(chosen, start=1):
        rel = path.relative_to(CIRQ_ROOT)
        print(f"[{idx}] {rel}")
        summarize_circuit(str(rel))
        print("-" * 80)


## 3) View One Circuit (Full)
Set `REL_PATH` and run to print all ops and show the built-in Cirq SVG rendering.

In [3]:
REL_PATH = "0/0.cirq.pkl"  # e.g., "29/8.cirq.pkl"
show_all_ops_and_view(REL_PATH)

File: const_0_noiseless_lrb_cirq\0\0.cirq.pkl
[ERROR] Failed to load pickle: 'type' object is not subscriptable


## 4) Random Spot-Check
Quickly summarize a few random circuits (without printing every operation).

In [4]:
preview_random(n=3)

[1] 16\3.cirq.pkl
File: const_0_noiseless_lrb_cirq\16\3.cirq.pkl
[ERROR] Failed to load pickle: 'type' object is not subscriptable
--------------------------------------------------------------------------------
[2] 4\2.cirq.pkl
File: const_0_noiseless_lrb_cirq\4\2.cirq.pkl
[ERROR] Failed to load pickle: 'type' object is not subscriptable
--------------------------------------------------------------------------------
[3] 26\6.cirq.pkl
File: const_0_noiseless_lrb_cirq\26\6.cirq.pkl


[ERROR] Failed to load pickle: 'type' object is not subscriptable
--------------------------------------------------------------------------------
